#🟦 01 - Setup
🔹 Paths and Configuration

In [0]:
# 🟦 01 - Setup
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col, trim, upper, lower, regexp_replace, to_date,
    when, row_number, count, sum as spark_sum, length
)

catalog = "workspace"
schema = "default"
volume = "raw_files"

PROJECT_NAME = "fuel_dashboard"

BRONZE_BASE = f"/Volumes/{catalog}/{schema}/{volume}/{PROJECT_NAME}/bronze"
SILVER_BASE = f"/Volumes/{catalog}/{schema}/{volume}/{PROJECT_NAME}/silver"

SOURCES = {
    "anp_prices": {
        "bronze": f"{BRONZE_BASE}/anp_prices",
        "silver": f"{SILVER_BASE}/anp_prices",
    },
    "anp_refinery": {
        "bronze": f"{BRONZE_BASE}/anp_refinery",
        "silver": f"{SILVER_BASE}/anp_refinery",
    },
    "anp_imports": {
        "bronze": f"{BRONZE_BASE}/anp_imports",
        "silver": f"{SILVER_BASE}/anp_imports",
    },
    "brent": {
        "bronze": f"{BRONZE_BASE}/brent",
        "silver": f"{SILVER_BASE}/brent",
    }
}

print("Bronze base:", BRONZE_BASE)
print("Silver base:", SILVER_BASE)
print("Sources loaded:", list(SOURCES.keys()))

#🟦 02 - Functions

In [0]:
# 🟦 02 - Functions
def clean_string(column_name):
    """
    Trim spaces, collapse repeated spaces, and convert empty strings to null.
    """
    c = F.trim(F.col(column_name).cast("string"))
    c = F.regexp_replace(c, r"\s+", " ")
    return F.when(c == "", None).otherwise(c)


def parse_decimal(column_name):
    """
    Convert Brazilian decimal strings to double.
    Examples:
    '5,79' -> 5.79
    '1.234,56' -> 1234.56
    '' -> null
    """
    c = F.trim(F.col(column_name).cast("string"))
    c = F.regexp_replace(c, ",", ".")    # decimal comma to dot
    return F.when(c == "", None).otherwise(c.cast("double"))

def null_count_expr(column_name):
    """
    Reusable helper for profiling nulls.
    """
    return F.sum(F.col(column_name).isNull().cast("int")).alias(f"null_{column_name}")

#🟦 03 - Transformations

In [0]:
# 🟦 03 - Transformations
# 3.1 Read data from bronze

df_prices = spark.read.format("delta").load(SOURCES["anp_prices"]["bronze"])
df_anp_refinery = spark.read.format("delta").load(SOURCES["anp_refinery"]["bronze"])
df_anp_imports = spark.read.format("delta").load(SOURCES["anp_imports"]["bronze"])
df_brent = spark.read.format("delta").load(SOURCES["brent"]["bronze"])

print("anp_prices rows:", df_prices.count())
print("anp_refinery rows:", df_anp_refinery.count())
print("anp_imports rows:", df_anp_imports.count())
print("brent rows:", df_brent.count())

#🟦 04 - Silver layer


## 4.1 anp_prices

In [0]:
#cleaning and casting
df_prices_silver = (
    df_prices
    .withColumn("regiao_sigla", upper(clean_string("regiao_sigla")))
    .withColumn("estado_sigla", upper(clean_string("estado_sigla")))
    .withColumn("municipio", clean_string("municipio"))
    .withColumn("revenda", clean_string("revenda"))
    .withColumn("cnpj_da_revenda", clean_string("cnpj_da_revenda"))
    .withColumn("produto", clean_string("produto"))
    .withColumn(
        "data_da_coleta",
        when(
            col("data_da_coleta").rlike(r"^\d{2}/\d{2}/\d{4}$"),
            to_date(col("data_da_coleta"), "dd/MM/yyyy")
        )
    )
    .withColumn("valor_de_venda", parse_decimal("valor_de_venda"))
    .withColumn("valor_de_compra", parse_decimal("valor_de_compra"))
    .withColumn("unidade_de_medida", clean_string("unidade_de_medida"))
    .withColumn("bandeira", clean_string("bandeira"))
)

In [0]:
df_prices_silver_nulls = df_prices_silver.agg(
    null_count_expr("data_da_coleta"),
    null_count_expr("produto"),
    null_count_expr("valor_de_venda")
)
display(df_prices_silver_nulls)


In [0]:
df_prices_silver = df_prices_silver.filter(
    col("data_da_coleta").isNotNull() &
    col("produto").isNotNull() &
    col("valor_de_venda").isNotNull()
)

In [0]:
df_prices_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema}.{PROJECT_NAME}_anp_prices_silver")

## 4.2 Silver layer for anp_refinery

In [0]:
df_anp_refinery_silver = (
    df_anp_refinery
    .withColumn("unidade_da_federacao", upper(clean_string("unidade_da_federacao")))
    .withColumn("refinaria", upper(clean_string("refinaria")))
    .withColumn("produto", clean_string("produto"))
    .withColumn("ano",when(col("ano").rlike(r"^\d{4}$"),col("ano").cast("int")))
    .withColumn("producao", parse_decimal("producao"))
)   

#clean null values
df_anp_refinery_silver = df_anp_refinery_silver.filter(
    col("ano").isNotNull() &
    col("producao").isNotNull()
)   


In [0]:
#write silver
df_anp_refinery_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema}.{PROJECT_NAME}_anp_refinery_silver")

## 4.3 Silver layer for anp_imports

In [0]:
#cleaning and casting
df_anp_imports_silver = (
    df_anp_imports
    .withColumn("ano", when(col("ano").rlike(r"^\d{4}$"), col("ano").cast("int")))
    .withColumn("produto", clean_string("produto"))
    .withColumn("operacao_comercial", clean_string("operacao_comercial"))
    .withColumn("importado_exportado", parse_decimal("importado_exportado"))
    .withColumn("dispendio_receita", parse_decimal("dispendio_receita"))
)

#clean null values
df_anp_imports_silver = df_anp_imports_silver.filter(
    col("ano").isNotNull() &
    col("produto").isNotNull() &
    col("importado_exportado").isNotNull()
)
df_anp_imports_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema}.{PROJECT_NAME}_anp_imports_silver")

## 4.4 Silver layer for Brent

In [0]:
df_brent_silver = (
    df_brent
    .withColumn("date", when(col("date").rlike(r"^\d{4}-\d{2}-\d{2}$"), to_date(col("date"), "yyyy-MM-dd")))
    .withColumn("price", parse_decimal("price"))
)

df_brent_silver = df_brent_silver.filter(
    col("date").isNotNull() &
    col("price").isNotNull()
)

df_brent_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema}.{PROJECT_NAME}_brent_silver")

#🟦 05 - Validations

## 5.1 anp_prices

In [0]:
total_rows = df_prices_silver.count()

valid_date_rows = df_prices_silver.filter(col("data_da_coleta").isNotNull()).count()
valid_price_rows = df_prices_silver.filter(col("valor_de_venda").isNotNull()).count()

print("=== DATA QUALITY SUMMARY: anp_prices ===")
print("total_rows:", total_rows)
print("valid_date_rows:", valid_date_rows)
print("invalid_date_rows:", total_rows - valid_date_rows)
print("valid_price_rows:", valid_price_rows)
print("invalid_price_rows:", total_rows - valid_price_rows)

In [0]:
print("Bronze:", df_prices.count())
print("Silver:", df_prices_silver.count())
print("Bronze - Silver:", df_prices.count()-df_prices_silver.count())

## 5.2 anp_refinery

In [0]:
total_rows = df_anp_refinery.count()

valid_date_rows = df_anp_refinery.filter(col("ano").isNotNull()).count()
valid_prod_rows = df_anp_refinery.filter(col("producao").isNotNull()).count()

print("=== DATA QUALITY SUMMARY: anp_refinery ===")
print("total_rows:", total_rows)
print("valid_date_rows:", valid_date_rows)
print("invalid_date_rows:", total_rows - valid_date_rows)
print("valid_price_rows:", valid_prod_rows)
print("invalid_price_rows:", total_rows - valid_prod_rows)

In [0]:
total_rows = df_anp_refinery_silver.count()

valid_date_rows = df_anp_refinery_silver.filter(col("ano").isNotNull()).count()
valid_price_rows = df_anp_refinery_silver.filter(col("producao").isNotNull()).count()

print("=== DATA QUALITY SUMMARY: anp_refinery ===")
print("total_rows:", total_rows)
print("valid_date_rows:", valid_date_rows)
print("invalid_date_rows:", total_rows - valid_date_rows)
print("valid_price_rows:", valid_price_rows)
print("invalid_price_rows:", total_rows - valid_price_rows)

In [0]:
print("Bronze:", df_anp_refinery.count())
print("Silver:", df_anp_refinery_silver.count())
print("Bronze - Silver:", df_anp_refinery.count()-df_anp_refinery_silver.count())

## 5.3 anp_imports

In [0]:
print("Bronze:", df_anp_imports.count())
print("Silver:", df_anp_imports_silver.count())
print("Bronze - Silver:", df_anp_imports.count() - df_anp_imports_silver.count())

##5.4 Brent

In [0]:
print("Bronze:", df_brent.count())
print("Silver:", df_brent_silver.count())
print("Bronze - Silver:", df_brent.count() - df_brent_silver.count())

In [0]:
display(df_brent_silver)

In [0]:
df_anp_imports_silver.select("produto").distinct().show()


In [0]:
df_anp_imports.select("produto").distinct().show()

In [0]:
df_anp_refinery_silver.select("produto").distinct().show()
